# Inference Serving Benchmark: HF `generate` vs vLLM (single T4)

Model: TinyLlama-1.1B-Chat (optionally with the SFT LoRA adapter merged in).
Prompts: real `age + instruction` samples from `yxpan/children_sft_dataset`, same system prompt as SFT.

1. KV cache memory math (GQA)
2. HF `generate` with `use_cache=False` vs `True`, plus static batching
3. vLLM (PagedAttention + continuous batching), concurrency sweep
4. vLLM prefix caching off vs on, with a shared long RAG-style context
5. When does the KV cache pay off? Decode step time with / without cache vs context length and batch size
6. Realistic output lengths: each request generates as many tokens as its reference answer, instead of a fixed 128

Sections 3 and 6 force every generation to exactly `MAX_NEW_TOKENS` (`min_new_tokens` in HF, `ignore_eos` in vLLM) to isolate per-token efficiency; sections 3c / 7b replay realistic lengths.

Runtime: Colab, GPU = T4. If `pip install vllm` asks to restart the session, restart and continue from the config cell.

In [ ]:
# Colab only. On the course platform, use the venv with a CUDA 12.8+ torch instead and skip this cell.
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm openai datasets matplotlib pandas
# If the vLLM server fails to start on T4 (attention backend errors), fall back to an older release:
# !pip install -q vllm==0.6.6.post1

## 0. Config

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = None      # path / HF repo of the SFT LoRA adapter (config B). None = base model
N_PROMPTS = 128
MAX_NEW_TOKENS = 128
CONCURRENCY = [1, 8, 32]
PORT = 8000
RESULTS = {}

In [ ]:
# Save everything (results json, vLLM logs, plots) to Drive so a runtime disconnect loses nothing
import os, json, shutil, time
import torch
RUN_ID = time.strftime('%Y%m%d_%H%M%S')
try:
    from google.colab import drive   # Colab: save to Drive
    drive.mount("/content/drive")
    OUT_DIR = f"/content/drive/MyDrive/llm_serving_runs/{RUN_ID}"
except ImportError:                  # course platform: set OUT_ROOT to the persistent storage path
    OUT_DIR = os.path.join(os.environ.get("OUT_ROOT", os.path.expanduser("~/llm_serving_runs")), RUN_ID)
os.makedirs(OUT_DIR, exist_ok=True)

def save_results():
    json.dump(RESULTS, open(f"{OUT_DIR}/serving_results.json", "w"), indent=2, default=str)

def save_file(path):
    if os.path.exists(path):
        shutil.copy(path, OUT_DIR)
# GPU sanity check + record hardware with every run (results differ across GPUs / MIG slices)
major, minor = torch.cuda.get_device_capability(0)
assert major >= 7, f"compute capability {major}.{minor}: vLLM needs >= 7.0, request another GPU"
torch.ones(4, device="cuda").sum().item()  # fails here if torch has no kernels for this GPU (e.g. Blackwell needs CUDA 12.8+ torch)
free, total = torch.cuda.mem_get_info(0)
RESULTS["env"] = dict(gpu=torch.cuda.get_device_name(0), compute=f"{major}.{minor}",
                      gpu_mem_gib=round(total / 2**30, 1), torch=torch.__version__, cuda=torch.version.cuda)
print(RESULTS["env"])
print("Outputs ->", OUT_DIR)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SERVE_PATH = MODEL_ID
if ADAPTER_PATH:
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    merged = PeftModel.from_pretrained(base, ADAPTER_PATH).merge_and_unload()
    merged.save_pretrained("merged_model")
    AutoTokenizer.from_pretrained(MODEL_ID).save_pretrained("merged_model")
    SERVE_PATH = "merged_model"
    del base, merged
print("Serving:", SERVE_PATH)

## 1. Prompts (same format as SFT)

In [ ]:
import random
from datasets import load_dataset

ds = load_dataset("yxpan/children_sft_dataset")
split = ds[list(ds.keys())[0]]
random.seed(0)
rows = random.sample(list(split), min(N_PROMPTS, len(split)))

def system_prompt(age):
    return (f"You are talking to a child aged {age}. "
            f"Generate a friendly response with age-appropriate knowledge.")

def to_messages(row):
    return [{"role": "system", "content": system_prompt(row["age"])},
            {"role": "user", "content": row["instruction"]}]

MESSAGES = [to_messages(r) for r in rows]
print(len(MESSAGES), MESSAGES[0])

## 2. KV cache memory math

In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained(SERVE_PATH)
L, H = cfg.num_hidden_layers, cfg.num_attention_heads
KVH = getattr(cfg, "num_key_value_heads", H)
D = cfg.hidden_size // H
BYTES = 2  # fp16

kv_per_tok = 2 * L * KVH * D * BYTES        # K and V
mha_per_tok = 2 * L * H * D * BYTES
print(f"layers={L}, q_heads={H}, kv_heads={KVH}, head_dim={D}")
print(f"KV cache per token: {kv_per_tok/1024:.1f} KB  (MHA would be {mha_per_tok/1024:.1f} KB, GQA saves {H//KVH}x)")
for ctx in [512, 2048]:
    print(f"one {ctx}-token sequence: {kv_per_tok*ctx/2**20:.1f} MiB")
RESULTS["kv_cache"] = dict(kb_per_token=kv_per_tok/1024, mha_kb_per_token=mha_per_tok/1024)

## 3. HF `generate` baseline

In [ ]:
import time, gc
import numpy as np

tok = AutoTokenizer.from_pretrained(SERVE_PATH)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(SERVE_PATH, torch_dtype=torch.float16).cuda().eval()

def hf_inputs(msgs_list):
    texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list]
    return tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")

@torch.no_grad()
def hf_run(msgs_list, use_cache, max_new_tokens=MAX_NEW_TOKENS):
    inputs = hf_inputs(msgs_list)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, min_new_tokens=max_new_tokens,
                         do_sample=False, use_cache=use_cache, pad_token_id=tok.pad_token_id)
    torch.cuda.synchronize()
    n_new = (out.shape[1] - inputs["input_ids"].shape[1]) * len(msgs_list)
    return time.perf_counter() - t0, n_new

hf_run(MESSAGES[:1], True, 8)  # warmup

N_HF = 8  # use_cache=False is slow, keep this small
for use_cache in [False, True]:
    ttft = np.mean([hf_run([m], use_cache, 1)[0] for m in MESSAGES[:N_HF]])
    lat, toks = zip(*[hf_run([m], use_cache) for m in MESSAGES[:N_HF]])
    tpot = (np.mean(lat) - ttft) / (MAX_NEW_TOKENS - 1)
    RESULTS[f"hf_cache={use_cache}_c1"] = dict(ttft_ms=ttft*1e3, tpot_ms=tpot*1e3, tok_per_s=sum(toks)/sum(lat))
    print(use_cache, RESULTS[f"hf_cache={use_cache}_c1"])

# static batching (whole batch waits for the longest sequence)
for bs in [b for b in CONCURRENCY if b > 1]:
    torch.cuda.reset_peak_memory_stats()
    lat, toks = hf_run(MESSAGES[:bs], True)
    RESULTS[f"hf_cache=True_bs{bs}"] = dict(tok_per_s=toks/lat, peak_mem_gb=torch.cuda.max_memory_allocated()/1e9)
    print(bs, RESULTS[f"hf_cache=True_bs{bs}"])

save_results()

## 3b. When does the KV cache pay off?

Batch-1 decode is bound by reading the weights once per step, so recomputing a short prefix (no cache) is almost free.
Recompute cost grows with context length and batch size, so the cache's speedup should grow with both.
Raw token ids, no chat template; TPOT = (time for `KV_OUT` tokens - time for 1 token) / (`KV_OUT` - 1).

In [ ]:
KV_LENS = [128, 512, 1024, 1920]   # prompt tokens; prompt + KV_OUT must stay <= 2048 (TinyLlama max positions)
KV_BS = [1, 8]
KV_OUT = 32
KV_REPS = 2

long_text = "\n\n".join(split.select(range(min(400, len(split))))["response"])
long_ids = tok(long_text, add_special_tokens=False).input_ids
assert len(long_ids) >= max(KV_LENS), len(long_ids)

@torch.no_grad()
def timed_generate(input_ids, use_cache, n_new):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    model.generate(input_ids=input_ids, attention_mask=torch.ones_like(input_ids),
                   max_new_tokens=n_new, min_new_tokens=n_new, do_sample=False,
                   use_cache=use_cache, pad_token_id=tok.pad_token_id)
    torch.cuda.synchronize()
    return time.perf_counter() - t0

KV_SWEEP = []
for bs in KV_BS:
    for L in KV_LENS:
        ids = torch.tensor([long_ids[:L]] * bs, device="cuda")
        row = dict(bs=bs, prompt_len=L)
        for use_cache in [True, False]:
            t1 = np.median([timed_generate(ids, use_cache, 1) for _ in range(KV_REPS)])
            tn = np.median([timed_generate(ids, use_cache, KV_OUT) for _ in range(KV_REPS)])
            row[f"tpot_ms_cache={use_cache}"] = (tn - t1) / (KV_OUT - 1) * 1e3
        row["speedup"] = row["tpot_ms_cache=False"] / row["tpot_ms_cache=True"]
        KV_SWEEP.append(row); print({k: round(v, 2) for k, v in row.items()})
RESULTS["kv_cache_sweep"] = KV_SWEEP
save_results()

## 3c. Realistic output lengths (HF static batching)

Each request generates as many tokens as its reference answer in the dataset (capped at `OUT_CAP`).
A static batch keeps running until its longest sequence finishes, so short requests waste compute.
Throughput counts only the tokens each request actually needs.

In [ ]:
OUT_CAP = 512
OUT_LENS = [max(2, min(len(tok(r["response"], add_special_tokens=False).input_ids), OUT_CAP)) for r in rows]
RESULTS["out_len_stats"] = dict(mean=float(np.mean(OUT_LENS)), p50=float(np.percentile(OUT_LENS, 50)),
                                p90=float(np.percentile(OUT_LENS, 90)), max=int(max(OUT_LENS)))
print("output lengths:", RESULTS["out_len_stats"])

REAL_BS = [8, 32]
for bs in REAL_BS:
    total_t, useful, generated = 0.0, 0, 0
    for s in range(0, len(MESSAGES), bs):
        lens = OUT_LENS[s:s + bs]
        t, n = hf_run(MESSAGES[s:s + bs], True, max(lens))  # whole batch runs to its longest request
        total_t += t; generated += n; useful += sum(lens)
    RESULTS[f"hf_real_bs{bs}"] = dict(tok_per_s=useful / total_t, wasted_frac=1 - useful / generated)
    print(bs, RESULTS[f"hf_real_bs{bs}"])
save_results()

In [ ]:
# free the HF model before starting vLLM
del model; gc.collect(); torch.cuda.empty_cache()

## 4. vLLM server helpers

In [ ]:
import subprocess, requests, re, asyncio
import sys, shutil
# use the vllm CLI from the same env as this kernel (venv bin may not be on PATH)
VLLM_BIN = os.path.join(os.path.dirname(sys.executable), "vllm")
if not os.path.exists(VLLM_BIN):
    VLLM_BIN = shutil.which("vllm") or "vllm"
from openai import AsyncOpenAI

def _launch(flags, logname):
    return subprocess.Popen(
        [VLLM_BIN, "serve", SERVE_PATH, "--dtype", "half", "--max-model-len", "2048",
         "--gpu-memory-utilization", "0.85", "--port", str(PORT),
         "--served-model-name", "tinyllama", *flags],
        stdout=open(logname, "w"), stderr=subprocess.STDOUT)

def prefix_caching_state(logname):
    m = re.search(r"enable_prefix_caching=(True|False)", open(logname).read())
    return None if m is None else m.group(1) == "True"

def start_server(prefix_caching, tag=""):
    logname = f"vllm_pc{int(prefix_caching)}{tag}.log"
    flags = ["--enable-prefix-caching"] if prefix_caching else ["--no-enable-prefix-caching"]
    proc = _launch(flags, logname)
    for _ in range(300):
        if proc.poll() is not None:
            log = open(logname).read()
            if not prefix_caching and "unrecognized arguments" in log:
                # older vLLM: no --no-enable-prefix-caching flag, and prefix caching is off by default
                flags = []; proc = _launch(flags, logname); continue
            save_file(logname)
            raise RuntimeError(log[-3000:])
        try:
            if requests.get(f"http://localhost:{PORT}/health", timeout=2).status_code == 200:
                state = prefix_caching_state(logname)
                print(f"server up: {logname}, flags={flags}, enable_prefix_caching in log = {state}")
                if state is not None and state != prefix_caching:
                    proc.terminate(); save_file(logname)
                    raise RuntimeError(f"asked prefix_caching={prefix_caching} but server reports {state}")
                return proc, logname
        except requests.exceptions.RequestException:
            pass
        time.sleep(2)
    save_file(logname)
    raise TimeoutError(open(logname).read()[-3000:])

def stop_server(proc, logname):
    proc.terminate(); proc.wait(60); time.sleep(5)
    save_file(logname)

def prefix_hit_rate(logname):
    hits = re.findall(r"[Pp]refix cache hit rate[^\d]*([\d.]+)%", open(logname).read())
    return hits[-1] + "%" if hits else None

def kv_capacity(logname):
    txt = open(logname).read()
    m = re.search(r"GPU KV cache size: ([\d,]+) tokens", txt) or re.search(r"# GPU blocks: (\d+)", txt)
    return m.group(0) if m else None

client = AsyncOpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="EMPTY")

async def one_request(messages, max_tokens):
    t0 = time.perf_counter(); ttft = None; n = 0
    stream = await client.chat.completions.create(
        model="tinyllama", messages=messages, max_tokens=max_tokens, temperature=0,
        stream=True, stream_options={"include_usage": True}, extra_body={"ignore_eos": True})
    async for chunk in stream:
        if ttft is None and chunk.choices and chunk.choices[0].delta.content:
            ttft = time.perf_counter() - t0
        if chunk.usage:
            n = chunk.usage.completion_tokens
    return ttft, time.perf_counter() - t0, n

async def run_bench(msgs_list, concurrency, max_tokens=MAX_NEW_TOKENS):
    # max_tokens: one int for all requests, or a list with one length per request
    lens = list(max_tokens) if isinstance(max_tokens, (list, tuple)) else [max_tokens] * len(msgs_list)
    sem = asyncio.Semaphore(concurrency)
    async def worker(m, n):
        async with sem:
            return await one_request(m, n)
    t0 = time.perf_counter()
    res = await asyncio.gather(*[worker(m, n) for m, n in zip(msgs_list, lens)])
    wall = time.perf_counter() - t0
    ttft, e2e, n = (np.array(x, dtype=float) for x in zip(*res))
    tpot = (e2e - ttft) / np.maximum(n - 1, 1)
    return dict(ttft_ms=ttft.mean()*1e3, ttft_p90_ms=np.percentile(ttft, 90)*1e3,
                tpot_ms=tpot.mean()*1e3, tok_per_s=n.sum()/wall)

## 5. Shared-prefix workload for prefix caching

The SFT system prompt alone is only ~25 tokens, so prefix caching barely matters for it.
To model RAG, every request shares the same ~1k-token reference block in the system message and asks a different question.

In [ ]:
_tok = AutoTokenizer.from_pretrained(SERVE_PATH)
ctx_rows = random.Random(1).sample(list(split), 40)
context = ""
for r in ctx_rows:
    context += r["response"].strip() + "\n\n"
    if len(_tok(context).input_ids) > 1000:
        break
AGE = rows[0]["age"]
PREFIX_MESSAGES = [
    [{"role": "system", "content": system_prompt(AGE) + "\nUse the reference material below.\n\n" + context},
     {"role": "user", "content": r["instruction"]}]
    for r in rows[:32]
]
print("shared prefix tokens:", len(_tok(PREFIX_MESSAGES[0][0]["content"]).input_ids))

## 6. vLLM run 1: prefix caching OFF (concurrency sweep + prefix baseline)

In [ ]:
proc, log0 = start_server(prefix_caching=False)
await run_bench(MESSAGES[:4], 4, 8)  # warmup

for c in CONCURRENCY:
    RESULTS[f"vllm_c{c}"] = await run_bench(MESSAGES, c)
    print(c, RESULTS[f"vllm_c{c}"])
    save_results()

await run_bench(PREFIX_MESSAGES[:1], 1, 4)  # same warmup as the ON run
RESULTS["prefix_off"] = await run_bench(PREFIX_MESSAGES, 1, 16)
RESULTS["vllm_kv_capacity"] = kv_capacity(log0)
RESULTS["prefix_off_hit_rate"] = prefix_hit_rate(log0)
print(RESULTS["prefix_off"], RESULTS["vllm_kv_capacity"], RESULTS["prefix_off_hit_rate"])
save_results()
stop_server(proc, log0)

## 7. vLLM run 2: prefix caching ON

In [ ]:
proc, log1 = start_server(prefix_caching=True)
await run_bench(PREFIX_MESSAGES[:1], 1, 4)  # first request fills the prefix cache
RESULTS["prefix_on"] = await run_bench(PREFIX_MESSAGES, 1, 16)
RESULTS["prefix_on_hit_rate"] = prefix_hit_rate(log1)
print(RESULTS["prefix_on"], RESULTS["prefix_on_hit_rate"])
save_results()
stop_server(proc, log1)

## 7b. vLLM with realistic output lengths

Same prompts and per-request lengths as 3c. Continuous batching lets a finished request leave the batch immediately.

In [ ]:
proc, log2 = start_server(prefix_caching=False, tag="_real")
await run_bench(MESSAGES[:4], 4, 8)  # warmup
for c in REAL_BS:
    RESULTS[f"vllm_real_c{c}"] = await run_bench(MESSAGES, c, OUT_LENS)
    print(c, RESULTS[f"vllm_real_c{c}"])
    save_results()
stop_server(proc, log2)

## 8. Results

In [ ]:
import json, pandas as pd
import matplotlib.pyplot as plt

save_results()
table = pd.DataFrame({k: v for k, v in RESULTS.items() if isinstance(v, dict) and k not in ("kv_cache", "env", "out_len_stats")}).T.round(2)
display(table)
table.to_csv(f"{OUT_DIR}/serving_results.csv")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
labels = ["HF no cache", "HF cache"]
vals = [RESULTS["hf_cache=False_c1"]["tpot_ms"], RESULTS["hf_cache=True_c1"]["tpot_ms"]]
ax.bar(labels, vals); ax.set_ylabel("ms / output token"); ax.set_title("Decode latency (batch 1)")

ax = axes[1]
x = np.arange(len(CONCURRENCY)); w = 0.38
hf = [RESULTS["hf_cache=True_c1"]["tok_per_s"]] + [RESULTS[f"hf_cache=True_bs{b}"]["tok_per_s"] for b in CONCURRENCY[1:]]
vl = [RESULTS[f"vllm_c{c}"]["tok_per_s"] for c in CONCURRENCY]
ax.bar(x - w/2, hf, w, label="HF static batch"); ax.bar(x + w/2, vl, w, label="vLLM")
ax.set_xticks(x, [str(c) for c in CONCURRENCY]); ax.set_xlabel("batch / concurrency")
ax.set_ylabel("output tokens / s"); ax.set_title("Throughput"); ax.legend()

ax = axes[2]
ax.bar(["prefix cache OFF", "prefix cache ON"], [RESULTS["prefix_off"]["ttft_ms"], RESULTS["prefix_on"]["ttft_ms"]])
ax.set_ylabel("mean TTFT (ms)"); ax.set_title("Shared ~1k-token prefix")

plt.tight_layout(); plt.savefig(f"{OUT_DIR}/serving_benchmark.png", dpi=150); plt.show()

## 9. KV cache sweep and realistic-length results

In [ ]:
kv = pd.DataFrame(RESULTS["kv_cache_sweep"])
display(kv.round(2))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
for bs, g in kv.groupby("bs"):
    ax.plot(g.prompt_len, g["tpot_ms_cache=False"], "o--", label=f"no cache, bs={bs}")
    ax.plot(g.prompt_len, g["tpot_ms_cache=True"], "o-", label=f"cache, bs={bs}")
ax.set_yscale("log"); ax.set_xlabel("prompt length (tokens)"); ax.set_ylabel("ms / decode step")
ax.set_title("Decode step time vs context"); ax.legend(fontsize=8)

ax = axes[1]
for bs, g in kv.groupby("bs"):
    ax.plot(g.prompt_len, g.speedup, "o-", label=f"bs={bs}")
ax.axhline(1, color="gray", lw=0.8)
ax.set_xlabel("prompt length (tokens)"); ax.set_ylabel("no-cache / cache")
ax.set_title("KV cache speedup"); ax.legend()

ax = axes[2]
x = np.arange(len(REAL_BS)); w = 0.2
series = [("HF, fixed 128", [RESULTS[f"hf_cache=True_bs{b}"]["tok_per_s"] for b in REAL_BS]),
          ("vLLM, fixed 128", [RESULTS[f"vllm_c{b}"]["tok_per_s"] for b in REAL_BS]),
          ("HF, realistic", [RESULTS[f"hf_real_bs{b}"]["tok_per_s"] for b in REAL_BS]),
          ("vLLM, realistic", [RESULTS[f"vllm_real_c{b}"]["tok_per_s"] for b in REAL_BS])]
for k, (name, vals) in enumerate(series):
    ax.bar(x + (k - 1.5) * w, vals, w, label=name)
ax.set_xticks(x, [str(b) for b in REAL_BS]); ax.set_xlabel("batch / concurrency")
ax.set_ylabel("useful output tokens / s"); ax.set_title("Fixed vs realistic output lengths"); ax.legend(fontsize=8)

plt.tight_layout(); plt.savefig(f"{OUT_DIR}/kv_and_realistic.png", dpi=150); plt.show()